# S3 → 로컬 conf/data 다운로드

`prepare_input/prepare.ipynb` 실행 후, S3에 업로드된 conf/data 파일을 로컬로 다운로드합니다.

In [ ]:
import os
import boto3
from pathlib import Path
from typing import List

In [ ]:
class InputConfig:
    REGION     = "us-east-1"
    ENV        = "dev"
    USER_ID    = "hjsong"
    PROJECT    = "bike-sharing-demand"
    EXPERIMENT = "baseline-lgbm-v1"
    VERSION    = "v1.0"

    CONF_BUCKET = f"gs-retail-awesome-conf-{REGION}"
    DATA_BUCKET = f"gs-retail-awesome-data-{REGION}"

    @classmethod
    def get_conf_prefix(cls):
        return f"{cls.ENV}/{cls.USER_ID}/{cls.PROJECT}/{cls.EXPERIMENT}"

    @classmethod
    def get_data_prefix(cls):
        return f"{cls.ENV}/{cls.USER_ID}/{cls.PROJECT}/{cls.VERSION}/data"

In [ ]:
def download_from_s3(bucket: str, s3_prefix: str, local_dir: str) -> List[str]:
    s3_client = boto3.client('s3', region_name=InputConfig.REGION)
    downloaded = []
    full_prefix = s3_prefix.rstrip('/') + '/'

    try:
        paginator = s3_client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=bucket, Prefix=full_prefix):
            for obj in page.get('Contents', []):
                s3_key = obj['Key']
                if s3_key.endswith('/'):
                    continue
                relative_path = s3_key[len(full_prefix):].lstrip('/')
                local_path = os.path.join(local_dir, relative_path)
                os.makedirs(os.path.dirname(local_path), exist_ok=True)
                s3_client.download_file(bucket, s3_key, local_path)
                print(f"  ✓ s3://{bucket}/{s3_key} -> {local_path}")
                downloaded.append(local_path)
    except Exception as e:
        print(f"✗ 다운로드 오류: {e}")
    return downloaded

In [ ]:
def download_inputs(local_root: str = "."):
    local_base     = Path(local_root)
    local_conf_dir = local_base / "conf"
    local_data_dir = local_base / "data"

    print("=" * 60)
    print(f"Step 0. 로컬 경로: {local_base.resolve()}")
    print("=" * 60)

    print("=" * 60)
    print("Step 1. Conf 다운로드")
    print(f"        s3://{InputConfig.CONF_BUCKET}/{InputConfig.get_conf_prefix()}/")
    print("=" * 60)
    conf_downloaded = download_from_s3(
        bucket    = InputConfig.CONF_BUCKET,
        s3_prefix = InputConfig.get_conf_prefix(),
        local_dir = str(local_conf_dir)
    )

    print("\n" + "=" * 60)
    print("Step 2. Data 다운로드")
    print(f"        s3://{InputConfig.DATA_BUCKET}/{InputConfig.get_data_prefix()}/")
    print("=" * 60)
    data_downloaded = download_from_s3(
        bucket    = InputConfig.DATA_BUCKET,
        s3_prefix = InputConfig.get_data_prefix(),
        local_dir = str(local_data_dir)
    )

    print("\n" + "=" * 60)
    print(f"다운로드 완료!")
    print(f"- Conf 파일 {len(conf_downloaded)}개 -> {local_conf_dir}")
    print(f"- Data 파일 {len(data_downloaded)}개 -> {local_data_dir}")
    print("=" * 60)

    return {
        "conf_dir": local_conf_dir,
        "data_dir": local_data_dir,
        "conf_files": conf_downloaded,
        "data_files": data_downloaded
    }

download_inputs()